# 07 Model Evaluation

In [ ]:
!rm -rf /content/unet-ensemble

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
sys.path.append('/content/unet-ensemble')

In [ ]:
!pip install safetensors huggingface_hub segmentation-models-pytorch scikit-image scipy -q

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from huggingface_hub import login
from torch.utils.data import DataLoader

from src.dataset.dataset     import Dataset as LazyDS
from src.dataset.dataloader  import DataLoader as DSLoader
from src.evaluation.evaluate import Evaluate

In [ ]:
token = ''
login(token=token)

## 07-1 Configuration

In [ ]:
IMG_SIZE     = 256
DATASET_ROOT = '/content/drive/Shareddrives/U-Net Ensemble/Dataset'
BATCH_SIZE   = 8
NUM_WORKERS  = 2

HF_USERNAME   = 'hf-username'
MODEL_REPO   = f'{HF_USERNAME}/mben-'

SUBFOLDER_MAP = {
    ('prnu', 'illumination', 'frequency') : 'all_features',
    ('prnu', 'frequency')                 : 'prnu_frequency',
    ('prnu', 'illumination')              : 'prnu_illumination',
    ('frequency', 'illumination')         : 'frequency_illumination',
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 07-2 Dataset Preparation

In [ ]:
MASK_FOLDER         = 'GTA - Masks'
PRNU_FOLDER         = 'Feature - PRNU'
ILLUMINATION_FOLDER = 'Feature - Illumination'
FREQUENCY_FOLDER    = 'Feature - Frequency'
CATEGORIES          = ['Synthetic', 'Authentic']
TEMPLATES           = [
    'template-a', 'template-albania', 'template-b',
    'template-c', 'template-chile',   'template-deutschland',
    'template-spain', 'template-usa',
]

In [ ]:
test_loaders = {}

for features in SUBFOLDER_MAP.keys():
    ds_loader = DSLoader(
        mask_folder         = MASK_FOLDER,
        prnu_folder         = PRNU_FOLDER,
        illumination_folder = ILLUMINATION_FOLDER,
        frequency_folder    = FREQUENCY_FOLDER,
        categories          = CATEGORIES,
        templates           = TEMPLATES,
        features            = features,
    )

    test_samples = ds_loader.load_images('Evaluation', DATASET_ROOT)
    test_ds      = LazyDS(test_samples, IMG_SIZE, augment=False, features=features)

    test_loaders[features] = DataLoader(
        test_ds,
        batch_size  = BATCH_SIZE,
        shuffle     = False,
        num_workers = NUM_WORKERS,
        pin_memory  = True,
    )

    print(f'{sorted(features)} → {len(test_loaders[features])} batches')

## 07-3 Evaluation Loop

In [ ]:
all_results = []  # list of dicts — one row per feature combination

for features, subfolder in SUBFOLDER_MAP.items():
    loader    = test_loaders[features]
    label     = '+'.join(sorted(features))
    evaluator = Evaluate(device=device, features=features)

    print(f'\n[Model] features: {sorted(features)}')
    try:
        model         = evaluator.load_unetpp_from_hub(MODEL_REPO, subfolder)  # ← swap loader method if needed
        model_metrics = evaluator.run(loader, model)
        Evaluate.print_metrics(model_metrics, label=label)
        all_results.append({'Features': label, **model_metrics})
        del model
        torch.cuda.empty_cache()
    except Exception as e:
        print(f'  ERROR: {e}')

## 07-4 Result Table

In [ ]:
results_df = pd.DataFrame(all_results)
col_order  = ['Features', 'IoU', 'Dice', 'Pixel_Accuracy', 'BF_Score']
results_df = results_df[col_order]

pd.set_option('display.float_format', '{:.4f}'.format)
display(results_df)

In [ ]:
csv_path = '/content/drive/Shareddrives/U-Net Ensemble/evaluation_results.csv'
results_df.to_csv(csv_path, index=False)
print(f'Results saved to {csv_path}')

## 07-5 Metrics Visualization

In [ ]:
METRICS_TO_PLOT = ['IoU', 'Dice', 'Pixel_Accuracy', 'BF_Score']
feature_combos  = results_df['Features'].tolist()

x         = np.arange(len(feature_combos))
bar_width = 0.5

fig, axes = plt.subplots(1, len(METRICS_TO_PLOT),
                         figsize=(5 * len(METRICS_TO_PLOT), 5),
                         sharey=False)

for ax, metric in zip(axes, METRICS_TO_PLOT):
    vals = results_df[metric].tolist()
    ax.bar(x, vals, bar_width)
    ax.set_title(metric.replace('_', ' '))
    ax.set_xticks(x)
    ax.set_xticklabels(feature_combos, rotation=30, ha='right', fontsize=8)
    ax.set_ylim(0, 1)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.suptitle('MBEN Model Evaluation', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('evaluation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()